## Mini Project: Food Delivery Order Management

In [1]:
import json
import os


FILE_NAME = "orders.json"


# ==========================================
# Order Class
# ==========================================

class Order:

    def __init__(
        self,
        order_id,
        customer_name,
        items,
        total_amount,
        status="Pending"
    ):
        self.order_id = order_id
        self.customer_name = customer_name
        self.items = items
        self.total_amount = total_amount
        self.status = status

    def to_dict(self):
        """Convert Order object into dictionary."""
        return {
            "order_id": self.order_id,
            "customer_name": self.customer_name,
            "items": self.items,
            "total_amount": self.total_amount,
            "status": self.status
        }

    @classmethod
    def from_dict(cls, data):
        """Create Order object from dictionary."""
        return cls(
            data["order_id"],
            data["customer_name"],
            data["items"],
            data["total_amount"],
            data.get("status", "Pending")
        )


# ==========================================
# Load Orders from JSON
# ==========================================

def load_orders():

    try:

        if not os.path.exists(FILE_NAME):
            print("No existing order file found.")
            print("Starting with an empty order list.\n")
            return []

        with open(FILE_NAME, "r") as file:
            data = json.load(file)

        orders = []

        for order_data in data:
            order = Order.from_dict(order_data)
            orders.append(order)

        print(f"{len(orders)} existing order(s) loaded successfully.\n")

        return orders

    except FileNotFoundError:

        print("Error: orders.json file was not found.")
        return []

    except json.JSONDecodeError:

        print("Error: orders.json contains invalid JSON.")
        return []


# ==========================================
# Save Orders to JSON
# ==========================================

def save_orders(orders):

    try:

        data = []

        for order in orders:
            data.append(order.to_dict())

        with open(FILE_NAME, "w") as file:
            json.dump(data, file, indent=4)

        print("Orders saved successfully.")

    except OSError as e:

        print(f"Error while saving orders: {e}")


# ==========================================
# Generate Order ID
# ==========================================

def generate_order_id(orders):

    if not orders:
        return "ORD001"

    highest_number = 0

    for order in orders:

        try:
            number = int(order.order_id.replace("ORD", ""))

            if number > highest_number:
                highest_number = number

        except ValueError:
            continue

    return f"ORD{highest_number + 1:03d}"


# ==========================================
# Place New Order
# ==========================================

def place_order(orders):

    print("\n========== PLACE NEW ORDER ==========")

    # Customer Name Validation
    while True:

        customer_name = input("Enter customer name: ").strip()

        if customer_name == "":
            print("Error: Customer name cannot be empty.")
        else:
            break

    # Items Input
    while True:

        items_input = input(
            "Enter items (comma-separated): "
        ).strip()

        if items_input == "":
            print("Error: Please enter at least one item.")
        else:

            items = [
                item.strip()
                for item in items_input.split(",")
                if item.strip()
            ]

            if len(items) == 0:
                print("Error: Please enter valid items.")
            else:
                break

    # Total Amount Validation
    while True:

        try:

            amount_input = input(
                "Enter total amount: ₹"
            ).strip()

            total_amount = float(amount_input)

            if total_amount <= 0:
                print("Error: Amount must be greater than 0.")
            else:
                break

        except ValueError:

            print(
                "Error: Total amount must be a numeric value."
            )

    # Generate Order ID
    order_id = generate_order_id(orders)

    # Create Order Object
    new_order = Order(
        order_id,
        customer_name,
        items,
        total_amount
    )

    # Add to list
    orders.append(new_order)

    # Save immediately
    save_orders(orders)

    print("\nOrder placed successfully!")
    print(f"Order ID: {order_id}")
    print(f"Customer: {customer_name}")
    print(f"Items: {', '.join(items)}")
    print(f"Total Amount: ₹{total_amount:.2f}")
    print(f"Status: {new_order.status}")


# ==========================================
# View All Orders
# ==========================================

def view_all_orders(orders):

    print("\n========== ALL ORDERS ==========")

    if not orders:
        print("No orders available.")
        return

    # Table Header
    print(
        f"{'Order ID':<10}"
        f"{'Customer Name':<20}"
        f"{'Items':<10}"
        f"{'Amount':<15}"
        f"{'Status':<15}"
    )

    print("-" * 70)

    for order in orders:

        # Highlight Delivered Orders
        if order.status.lower() == "delivered":

            print(
                f"✓ {order.order_id:<8}"
                f"{order.customer_name:<20}"
                f"{len(order.items):<10}"
                f"₹{order.total_amount:<14.2f}"
                f"{order.status:<15}"
            )

        else:

            print(
                f"{order.order_id:<10}"
                f"{order.customer_name:<20}"
                f"{len(order.items):<10}"
                f"₹{order.total_amount:<14.2f}"
                f"{order.status:<15}"
            )

    print("-" * 70)
    print("✓ = Delivered Order")


# ==========================================
# Search Order
# ==========================================

def search_order(orders):

    print("\n========== SEARCH ORDER ==========")

    order_id = input(
        "Enter Order ID: "
    ).strip().upper()

    if order_id == "":
        print("Error: Order ID cannot be empty.")
        return

    for order in orders:

        if order.order_id.upper() == order_id:

            print("\nOrder Found!")
            print("-" * 40)
            print(f"Order ID       : {order.order_id}")
            print(f"Customer Name  : {order.customer_name}")
            print(f"Items          : {', '.join(order.items)}")
            print(f"Item Count     : {len(order.items)}")
            print(f"Total Amount   : ₹{order.total_amount:.2f}")
            print(f"Status         : {order.status}")
            print("-" * 40)

            return

    print(f"No order found with ID: {order_id}")


# ==========================================
# Main Program
# ==========================================

def main():

    print("=" * 50)
    print("     FOOD DELIVERY ORDER MANAGEMENT")
    print("=" * 50)

    # Load existing orders when program starts
    orders = load_orders()

    while True:

        print("\n========== MAIN MENU ==========")
        print("1. Place New Order")
        print("2. View All Orders")
        print("3. Search Order by ID")
        print("4. Exit")

        choice = input("Enter your choice: ").strip()

        if choice == "1":

            place_order(orders)

        elif choice == "2":

            view_all_orders(orders)

        elif choice == "3":

            search_order(orders)

        elif choice == "4":

            print("\nSaving orders before exit...")
            save_orders(orders)

            print("Thank you for using the system!")
            print("Program closed.")
            break

        else:

            print(
                "Invalid choice. Please select "
                "a number between 1 and 4."
            )


# ==========================================
# Start Program
# ==========================================

main()

     FOOD DELIVERY ORDER MANAGEMENT
No existing order file found.
Starting with an empty order list.


========== MAIN MENU ==========
1. Place New Order
2. View All Orders
3. Search Order by ID
4. Exit

========== PLACE NEW ORDER ==========
Orders saved successfully.

Order placed successfully!
Order ID: ORD001
Customer: Shyam Patel
Items: Pizza, Burgar
Total Amount: ₹450.00
Status: Pending

========== MAIN MENU ==========
1. Place New Order
2. View All Orders
3. Search Order by ID
4. Exit

========== PLACE NEW ORDER ==========
Orders saved successfully.

Order placed successfully!
Order ID: ORD002
Customer: Ankit mepani
Items: Choclate-milk-shake, Badam-Shake
Total Amount: ₹250.00
Status: Pending

========== MAIN MENU ==========
1. Place New Order
2. View All Orders
3. Search Order by ID
4. Exit

Saving orders before exit...
Orders saved successfully.
Thank you for using the system!
Program closed.
